COMMANDS TO RUN OLLAMA LOCALLY

In [13]:
!sudo apt update

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
82 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRele

In [14]:
!sudo apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 82 not upgraded.


In [15]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [16]:
!pkill ollama
!ollama serve > ollama.log 2>&1 &

In [17]:
!ollama pull llama3.2

In [18]:
!ollama list

NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [19]:
!pip install pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("3A7AhlJ9UaKsOP0US5wciVnOl0I_3pDmajsssCbWAvHf7ov5W")

In [20]:
from pyngrok import ngrok
tunnel = ngrok.connect(11434, "http")
print("Forwarding URL:", tunnel.public_url)

Forwarding URL: https://versicolor-nonshedding-leola.ngrok-free.dev


if the above doesnt work, then try this

In [21]:
# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh
    
# 2. Start Ollama in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(5) # Wait for it to start
    
# 3. Pull the model
!ollama pull llama3.2

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%#####################################                 80.2%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [22]:
from pyngrok import ngrok
# Ensure any old tunnels are closed
ngrok.kill()
# Open tunnel to 11434 (Ollama's default port)
public_url = ngrok.connect(11434).public_url
print(f"New working URL: {public_url}")

New working URL: https://june-unshored-lavelle.ngrok-free.dev


In [23]:
import os
import subprocess
import time
    
# 1. Stop any running Ollama instance
!pkill ollama
    
# 2. ALLOW EXTERNAL ORIGINS (This fixes the 403 Forbidden error)
os.environ["OLLAMA_ORIGINS"] = "*"
   
# 3. Start Ollama with the new permission
subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # Give it a moment to start
   
# 4. Verify it's running locally
!curl http://localhost:11434/api/version

{"version":"0.17.0"}

In [24]:
import os
import subprocess
import time
    
print("🛑 Stopping existing Ollama...")
!pkill ollama
time.sleep(2)
    
print("🚀 Starting Ollama with allowed origins...")
# Set the environment variable explicitly for the new process
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
   
# Start the server in the background
subprocess.Popen(["ollama", "serve"])
   
print("⏳ Waiting for startup (5s)...")
time.sleep(5)
   
print("✅ Verifying local connection...")
!curl -v http://localhost:11434/api/version

🛑 Stopping existing Ollama...
🚀 Starting Ollama with allowed origins...
⏳ Waiting for startup (5s)...
✅ Verifying local connection...
*   Trying 127.0.0.1:11434...
* Connected to localhost (127.0.0.1) port 11434 (#0)
> GET /api/version HTTP/1.1
> Host: localhost:11434
> User-Agent: curl/7.81.0
> Accept: */*
> 
* Mark bundle as not supporting multiuse
< HTTP/1.1 200 OK
< Content-Type: application/json; charset=utf-8
< Date: Tue, 24 Feb 2026 11:47:23 GMT
< Content-Length: 20
< 
* Connection #0 to host localhost left intact
{"version":"0.17.0"}

you can test ollama locally by creating a file PyCode/test_ollama_collection.py in which you can write ->

import urllib.request
from urllib.error import URLError, HTTPError
import os
from dotenv import load_dotenv


load_dotenv("src/.env")

url = os.getenv("OLLAMA_BASE_URL")
print(f"Testing connection to: {url}")

if not url:
    print("❌ OLLAMA_BASE_URL is not set in src/.env")
    exit()


health_endpoint = f"{url.rstrip('/')}/api/version"
print(f"Probe Endpoint: {health_endpoint}")

req = urllib.request.Request(
    health_endpoint,
    headers={
        "ngrok-skip-browser-warning": "true",
        "User-Agent": "IntelliWrite-Test-Probe"
    }
)

try:
    with urllib.request.urlopen(req, timeout=10) as response:
        print(f"✅ Success! Status Code: {response.status}")
        print(f"Response: {response.read().decode('utf-8')}")
except HTTPError as e:
    print(f"❌ HTTP Error: {e.code} - {e.reason}")
    print(f"Headers: {e.headers}")
except URLError as e:
    print(f"❌ Connection Error: {e.reason}")
except Exception as e:
    print(f"❌ Unexpected Error: {e}") 

    and then in terminal run python test_ollama_connection.py  